<a href="https://colab.research.google.com/github/COMP3608-Group-12/Project/blob/Redo-Decision-Tree-Model-using-K-Fold-Cross-Validation---Dataset-2/Group_12_Dataset_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Initial Test Commit

## STEP 1: Importing Packages

In [ ]:
#importing packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import imblearn

## STEP 2: Load & Understand Data

In [ ]:
#loading csv file into pandas dataframe
df = pd.read_csv('card_transdata.csv')

#printing dataset information (how many rows, columns etc)
print(df.info())
print('----------------------------------------------------------')

#checking if any values are missing
print(df.isnull().sum())
print('----------------------------------------------------------')

#printing first 5 rows of data to ensure it correlates with datafile
print(df.head(5))
print('----------------------------------------------------------')

#checking how many transactions are fraudulent and how many are non-fraudulent
print(df['fraud'].value_counts())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 8 columns):
 #   Column                          Non-Null Count    Dtype  
---  ------                          --------------    -----  
 0   distance_from_home              1000000 non-null  float64
 1   distance_from_last_transaction  1000000 non-null  float64
 2   ratio_to_median_purchase_price  1000000 non-null  float64
 3   repeat_retailer                 1000000 non-null  int64  
 4   used_chip                       1000000 non-null  int64  
 5   used_pin_number                 1000000 non-null  int64  
 6   online_order                    1000000 non-null  int64  
 7   fraud                           1000000 non-null  int64  
dtypes: float64(3), int64(5)
memory usage: 61.0 MB
None
----------------------------------------------------------
distance_from_home                0
distance_from_last_transaction    0
ratio_to_median_purchase_price    0
repeat_retailer                   0


## Step 3: Split Data into Features & Target

In [ ]:
#Splitting features and target
X = df.drop('fraud', axis=1)
y = df['fraud']

## Step 4: Decision Tree Model using K Fold Cross Validation


In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from imblearn.over_sampling import SMOTE
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix

#Decision Tree Model
model_dt = DecisionTreeClassifier(random_state=42)

# StratifiedKFold
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Store results
all_results = []

# Manual KFold Loop
for fold, (train_index, test_index) in enumerate(kf.split(X, y), 1):

    print(f"\n================== Fold {fold} ==================")

    # Split
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # SMOTE on training data only
    smote = SMOTE(random_state=42)
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

    # Train model
    model_dt.fit(X_train_resampled, y_train_resampled)

    # Predict on test data
    y_pred = model_dt.predict(X_test)
    y_prob = model_dt.predict_proba(X_test)[:, 1]

    #Performance metrics
    results = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1 Score': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_prob)
    }

    results_df = pd.DataFrame({
        "Metric": list(results.keys()),
        "Value": [round(value, 4) for value in results.values()]
    })

    print(results_df)

    print(f"\n============================================")
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    print(f"\n============================================")
    print("Classification Report:")
    print(classification_report(y_test, y_pred, digits=4))

    # Appending results
    all_results.append(results)

# Taking average of all results
results_df = pd.DataFrame(all_results)

avg_results = results_df.mean()

final_df = pd.DataFrame({
    "Metric": avg_results.index,
    "Average Value": [round(val, 4) for val in avg_results.values]
})

print("\n================== AVERAGE PERFORMANCE ==================")
print(final_df)


================== Fold 1 ==================
      Metric   Value
0   Accuracy  0.9992
1  Precision  0.9910
2     Recall  0.9998
3   F1 Score  0.9954
4    ROC-AUC  0.9995

Confusion Matrix:
[[182362    158]
 [     3  17477]]

Classification Report:
              precision    recall  f1-score   support

           0     1.0000    0.9991    0.9996    182520
           1     0.9910    0.9998    0.9954     17480

    accuracy                         0.9992    200000
   macro avg     0.9955    0.9995    0.9975    200000
weighted avg     0.9992    0.9992    0.9992    200000


================== Fold 2 ==================
      Metric   Value
0   Accuracy  0.9991
1  Precision  0.9895
2     Recall  1.0000
3   F1 Score  0.9947
4    ROC-AUC  0.9995

Confusion Matrix:
[[182334    186]
 [     0  17480]]

Classification Report:
              precision    recall  f1-score   support

           0     1.0000    0.9990    0.9995    182520
           1     0.9895    1.0000    0.9947     17480

    accur

## Step 6: Deep Neural Network

### Step 6a: Defining Deep Neural Network Model

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.layers import Input # Added this line

def create_dnn_model(input_dim):
  model = models.Sequential([
    Input(shape = (input_dim,)),

    layers.Dense(32, activation = 'relu'),
    layers.BatchNormalization(),                                                    #Normalize outputs

    layers.Dense(16, activation = 'relu'),
    layers.Dropout(0.2),                                                           #Prevents overfitting (20% dropoff)

    layers.Dense(8, activation = 'relu'),

    layers.Dense(1, activation = 'sigmoid')
  ])

  model.compile(
      optimizer = tf.keras.optimizers.Adam(learning_rate = 0.001),
      loss = 'binary_crossentropy',
      metrics = [
          tf.keras.metrics.Precision(),
          tf.keras.metrics.Recall(),
          tf.keras.metrics.AUC()
      ]
  )
  return model

###Step 6b: Training Deep Neural Network Model

In [ ]:
#Training Deep Neural Network Model
model = create_dnn_model(input_dim = X_train.shape[1])

history = model.fit(
    X_train, y_train,
    epochs = 10,
    batch_size = 256,
    validation_split = 0.2
)

Epoch 1/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 12s 4ms/step - auc: 0.9821 - loss: 0.0928 - precision: 0.8273 - recall: 0.7320 - val_auc: 0.9985 - val_loss: 0.0314 - val_precision: 0.9860 - val_recall: 0.8966
Epoch 2/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - auc: 0.9969 - loss: 0.0360 - precision: 0.9201 - recall: 0.9206 - val_auc: 0.9991 - val_loss: 0.0220 - val_precision: 0.9493 - val_recall: 0.9516
Epoch 3/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - auc: 0.9978 - loss: 0.0291 - precision: 0.9378 - recall: 0.9342 - val_auc: 0.9930 - val_loss: 0.0388 - val_precision: 0.9518 - val_recall: 0.9070
Epoch 4/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 9s 4ms/step - auc: 0.9979 - loss: 0.0272 - precision: 0.9436 - recall: 0.9355 - val_auc: 0.9993 - val_loss: 0.0196 - val_precision: 0.9597 - val_recall: 0.9636
Epoch 5/10
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 9s 4ms/step - auc: 0.9983 - loss: 0.0246 - precision: 0.9500 - recall: 0.9400 - val_auc: 0.9997 - val_loss: 0.0133 - val_precision: 0.9862 - val_rec

### Step 6c: Making Predictions on Test Data

In [ ]:
y_pred = (model.predict(X_test) > 0.5).astype(float)
y_pred = y_pred.flatten()
y_prob = model.predict(X_test)
y_prob = y_prob.flatten()
print(y_pred.shape)
print(y_pred[:15])
print(y_prob.shape)
print(y_prob)

6250/6250 ━━━━━━━━━━━━━━━━━━━━ 7s 1ms/step
6250/6250 ━━━━━━━━━━━━━━━━━━━━ 7s 1ms/step
(200000,)
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
(200000,)
[7.2727807e-06 4.0872566e-10 3.9825072e-09 ... 1.4795856e-11 1.7480584e-08
 2.9429862e-07]


## Step 7: Logistic Regression

###Step 7a: Train the Logistic Regression Model

In [ ]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_resampled, y_train_resampled)

###Step 7b: Making Predictions

In [ ]:
#Predictions
y_pred = lr_model.predict(X_test)
y_prob = lr_model.predict_proba(X_test)[:, 1]

###Step 7c: Evaluating the Model Using Performance Metrics

In [ ]:
#Evaluating the model using performance metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix

#Calculating performance metrics
results = {
    'Accuracy': accuracy_score(y_test, y_pred),
    'Precision': precision_score(y_test, y_pred),
    'Recall': recall_score(y_test, y_pred),
    'F1 Score': f1_score(y_test, y_pred),
    'ROC-AUC': roc_auc_score(y_test, y_prob)
}

#Printing results in an easy to read table
results_df = pd.DataFrame({"Metric": list(results.keys()), "Value": [round(value, 4) for value in results.values()]})
print(results_df)

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("Classification Report:")
print(classification_report(y_test, y_pred, digits=4))